# Week 14 — QuakeMigrate Trigger stage
First inspects our own real coalescence function output to inform a
data-grounded starting threshold, rather than borrowing a value tuned for
a different dataset (icequakes or a different Iceland volcano) from
QuakeMigrate's bundled examples. Then runs Trigger using the confirmed
real API (template_scripts/trigger.py, cross-checked against the
Volcanotectonic_Iceland example - the closest real-world analogue to the
TVZ setting).

In [ ]:
import os
import numpy as np
import obspy

PROJECT_DIR = os.path.expanduser("~/tvz_project")
SCANMSEED_DIR = os.path.join(PROJECT_DIR, "quakemigrate", "runs", "tvz_full", "detect", "scanmseed")

# Inspect a sample of real days (not all 60, for a quick informative look)
sample_files = sorted(os.listdir(SCANMSEED_DIR))[::10]  # every 10th day
print(f"Inspecting {len(sample_files)} sample days out of {len(os.listdir(SCANMSEED_DIR))}")

all_coa = []
for fname in sample_files:
    st = obspy.read(os.path.join(SCANMSEED_DIR, fname))
    coa_trace = st.select(station="COA")
    if len(coa_trace) > 0:
        all_coa.append(coa_trace[0].data)

all_coa = np.concatenate(all_coa)
print(f"\nCOA value distribution across {len(sample_files)} sample days ({len(all_coa):,} samples):")
for pct in [50, 90, 95, 99, 99.9, 99.99, 100]:
    print(f"  {pct:>6}th percentile: {np.percentile(all_coa, pct):.3f}")
print(f"  Mean: {all_coa.mean():.3f}, Std: {all_coa.std():.3f}, Max: {all_coa.max():.3f}")

In [ ]:
# Diagnostic: check the real channel names actually present, rather than
# assume "COA" was correct.
sample_path = os.path.join(SCANMSEED_DIR, sorted(os.listdir(SCANMSEED_DIR))[0])
st = obspy.read(sample_path)
print(st)
print("\nChannels present:", sorted({tr.stats.channel for tr in st}))